# Modelo logistico 2026

Este notebook entrena y evalua una regresion logistica para clasificar si un evento armado de 2026 tiene fatalidades reportadas.

In [1]:
from pathlib import Path
import sys

import pandas as pd

BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

from scripts.train_logreg_2026 import (
    TARGET_COLUMN,
    coefficients_frame,
    evaluate_split,
    feature_columns,
    load_2026_data,
    make_pipeline,
    split_random,
    split_temporal,
)

df = load_2026_data()
df.shape

(1632, 65)

## Balance de clases

In [2]:
df[TARGET_COLUMN].value_counts(normalize=False).rename(index={0: 'no_letal', 1: 'letal'}), df[TARGET_COLUMN].mean()

(fatality_positive
 no_letal    1256
 letal        376
 Name: count, dtype: int64,
 np.float64(0.23039215686274508))

## Splits

La validacion temporal es la referencia principal. El split aleatorio estratificado se usa como diagnostico de senal interna.

In [3]:
feature_cols, numeric_cols, categorical_cols = feature_columns(df)
temporal_train, temporal_test = split_temporal(df)
random_train, random_test = split_random(df)

split_summary = pd.DataFrame(
    [
        {'split': 'temporal_train', 'rows': len(temporal_train), 'positive_rate': temporal_train[TARGET_COLUMN].mean()},
        {'split': 'temporal_test_may', 'rows': len(temporal_test), 'positive_rate': temporal_test[TARGET_COLUMN].mean()},
        {'split': 'random_train', 'rows': len(random_train), 'positive_rate': random_train[TARGET_COLUMN].mean()},
        {'split': 'random_test', 'rows': len(random_test), 'positive_rate': random_test[TARGET_COLUMN].mean()},
    ]
)
split_summary

,split,rows,positive_rate
0,temporal_train,1419,0.233968
1,temporal_test_may,213,0.206573
2,random_train,1305,0.230651
3,random_test,327,0.229358


## Entrenamiento

In [4]:
variants = [
    {'name': 'logreg_l2_balanced', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 1.0, 'l1_ratio': 0.0},
    {'name': 'logreg_l2_unweighted', 'feature_profile': 'full', 'class_weight': None, 'c_value': 1.0, 'l1_ratio': 0.0},
    {'name': 'logreg_l1_balanced_c1', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 1.0, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c1', 'feature_profile': 'full', 'class_weight': None, 'c_value': 1.0, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_balanced_c03', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 0.3, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c03', 'feature_profile': 'full', 'class_weight': None, 'c_value': 0.3, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_balanced_c01', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c01', 'feature_profile': 'full', 'class_weight': None, 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_core_l1_balanced_c01', 'feature_profile': 'core_interpretable', 'class_weight': 'balanced', 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_core_l1_unweighted_c01', 'feature_profile': 'core_interpretable', 'class_weight': None, 'c_value': 0.1, 'l1_ratio': 1.0},
]

metrics = []
predictions = []
models = {}
model_configs = {}
model_feature_specs = {}

for config in variants:
    variant = config['name']
    feature_profile = config['feature_profile']
    feature_cols, numeric_cols, categorical_cols = feature_columns(df, feature_profile)
    temporal_model = make_pipeline(
        numeric_cols,
        categorical_cols,
        config['class_weight'],
        config['c_value'],
        config['l1_ratio'],
    )
    temporal_metrics, temporal_predictions = evaluate_split(
        'temporal_train_until_apr_test_may',
        variant,
        feature_profile,
        temporal_model,
        temporal_train,
        temporal_test,
        feature_cols,
    )
    temporal_metrics['c_value'] = config['c_value']
    temporal_metrics['l1_ratio'] = config['l1_ratio']
    temporal_metrics['class_weight'] = config['class_weight'] or 'none'
    metrics.append(temporal_metrics)
    predictions.append(temporal_predictions)
    models[variant] = temporal_model
    model_configs[variant] = config
    model_feature_specs[variant] = (numeric_cols, categorical_cols)

    random_model = make_pipeline(
        numeric_cols,
        categorical_cols,
        config['class_weight'],
        config['c_value'],
        config['l1_ratio'],
    )
    random_metrics, random_predictions = evaluate_split(
        'random_stratified_80_20',
        variant,
        feature_profile,
        random_model,
        random_train,
        random_test,
        feature_cols,
    )
    random_metrics['c_value'] = config['c_value']
    random_metrics['l1_ratio'] = config['l1_ratio']
    random_metrics['class_weight'] = config['class_weight'] or 'none'
    metrics.append(random_metrics)
    predictions.append(random_predictions)

metrics_df = pd.DataFrame(metrics)
predictions_df = pd.concat(predictions, ignore_index=True)

metrics_df

,split,variant,feature_profile,train_rows,test_rows,train_positive_rate,test_positive_rate,roc_auc,average_precision,brier_score,...,fp,fn,tp,total_features_after_encoding,nonzero_features,zeroed_features,nonzero_feature_rate,c_value,l1_ratio,class_weight
0,temporal_train_until_apr_test_may,logreg_l2_balanced,full,1419,213,0.233968,0.206573,0.739510,0.422204,0.445457,...,140,1,43,157,155,2,0.987261,1.0,0.0,balanced
1,random_stratified_80_20,logreg_l2_balanced,full,1305,327,0.230651,0.229358,0.787989,0.577604,0.173789,...,60,20,55,148,146,2,0.986486,1.0,0.0,balanced
2,temporal_train_until_apr_test_may,logreg_l2_unweighted,full,1419,213,0.233968,0.206573,0.716649,0.386094,0.279420,...,93,5,39,157,155,2,0.987261,1.0,0.0,none
3,random_stratified_80_20,logreg_l2_unweighted,full,1305,327,0.230651,0.229358,0.788466,0.582067,0.134545,...,10,44,31,148,146,2,0.986486,1.0,0.0,none
4,temporal_train_until_apr_test_may,logreg_l1_balanced_c1,full,1419,213,0.233968,0.206573,0.743007,0.424378,0.409905,...,134,1,43,157,90,67,0.573248,1.0,1.0,balanced
5,random_stratified_80_20,logreg_l1_balanced_c1,full,1305,327,0.230651,0.229358,0.796667,0.583943,0.172890,...,60,20,55,148,86,62,0.581081,1.0,1.0,balanced
6,temporal_train_until_apr_test_may,logreg_l1_unweighted_c1,full,1419,213,0.233968,0.206573,0.718666,0.385702,0.241790,...,69,9,35,157,83,74,0.528662,1.0,1.0,none
7,random_stratified_80_20,logreg_l1_unweighted_c1,full,1305,327,0.230651,0.229358,0.793069,0.587743,0.133008,...,11,44,31,148,81,67,0.547297,1.0,1.0,none
8,temporal_train_until_apr_test_may,logreg_l1_balanced_c03,full,1419,213,0.233968,0.206573,0.722566,0.426441,0.279292,...,90,5,39,157,56,101,0.356688,0.3,1.0,balanced
9,random_stratified_80_20,logreg_l1_balanced_c03,full,1305,327,0.230651,0.229358,0.782963,0.555122,0.182336,...,66,23,52,148,53,95,0.358108,0.3,1.0,balanced


## Matriz de confusion

In [5]:
metrics_df[[
    'split', 'variant', 'c_value', 'l1_ratio', 'class_weight',
    'tn', 'fp', 'fn', 'tp', 'precision', 'recall', 'f1',
    'roc_auc', 'average_precision', 'brier_score',
    'nonzero_features', 'zeroed_features'
]]

,split,variant,c_value,l1_ratio,class_weight,tn,fp,fn,tp,precision,recall,f1,roc_auc,average_precision,brier_score,nonzero_features,zeroed_features
0,temporal_train_until_apr_test_may,logreg_l2_balanced,1.0,0.0,balanced,29,140,1,43,0.234973,0.977273,0.378855,0.739510,0.422204,0.445457,155,2
1,random_stratified_80_20,logreg_l2_balanced,1.0,0.0,balanced,192,60,20,55,0.478261,0.733333,0.578947,0.787989,0.577604,0.173789,146,2
2,temporal_train_until_apr_test_may,logreg_l2_unweighted,1.0,0.0,none,76,93,5,39,0.295455,0.886364,0.443182,0.716649,0.386094,0.279420,155,2
3,random_stratified_80_20,logreg_l2_unweighted,1.0,0.0,none,242,10,44,31,0.756098,0.413333,0.534483,0.788466,0.582067,0.134545,146,2
4,temporal_train_until_apr_test_may,logreg_l1_balanced_c1,1.0,1.0,balanced,35,134,1,43,0.242938,0.977273,0.389140,0.743007,0.424378,0.409905,90,67
5,random_stratified_80_20,logreg_l1_balanced_c1,1.0,1.0,balanced,192,60,20,55,0.478261,0.733333,0.578947,0.796667,0.583943,0.172890,86,62
6,temporal_train_until_apr_test_may,logreg_l1_unweighted_c1,1.0,1.0,none,100,69,9,35,0.336538,0.795455,0.472973,0.718666,0.385702,0.241790,83,74
7,random_stratified_80_20,logreg_l1_unweighted_c1,1.0,1.0,none,241,11,44,31,0.738095,0.413333,0.529915,0.793069,0.587743,0.133008,81,67
8,temporal_train_until_apr_test_may,logreg_l1_balanced_c03,0.3,1.0,balanced,79,90,5,39,0.302326,0.886364,0.450867,0.722566,0.426441,0.279292,56,101
9,random_stratified_80_20,logreg_l1_balanced_c03,0.3,1.0,balanced,186,66,23,52,0.440678,0.693333,0.538860,0.782963,0.555122,0.182336,53,95


## Coeficientes interpretables

Los coeficientes se calculan sobre el modelo temporal, que es la validacion principal.

In [6]:
coefs = pd.concat(
    [
        coefficients_frame(
            variant,
            model,
            model_feature_specs[variant][0],
            model_feature_specs[variant][1],
            model_configs[variant],
        )
        for variant, model in models.items()
    ],
    ignore_index=True,
)

coefs.sort_values('coefficient', ascending=False).head(20)

,variant,feature_profile,c_value,l1_ratio,class_weight,feature,coefficient,odds_ratio,abs_coefficient,selected_l1,numeric_feature_count,categorical_feature_count
315,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__country_Sri Lanka,2.354493,10.532787,2.354493,True,46,12
316,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__country_Lebanon,2.119829,8.329716,2.119829,True,46,12
471,logreg_l1_unweighted_c1,full,1.0,1.0,none,cat__country_Lebanon,2.105714,8.212966,2.105714,True,46,12
473,logreg_l1_unweighted_c1,full,1.0,1.0,none,cat__country_Sri Lanka,2.020581,7.542705,2.020581,True,46,12
319,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__sub_event_type_Shelling/artillery/missile...,1.769969,5.870669,1.769969,True,46,12
1,logreg_l2_balanced,full,1.0,0.0,balanced,cat__country_Lebanon,1.765339,5.843551,1.765339,True,46,12
2,logreg_l2_balanced,full,1.0,0.0,balanced,cat__country_Sri Lanka,1.762616,5.827663,1.762616,True,46,12
157,logreg_l2_unweighted,full,1.0,0.0,none,cat__country_Lebanon,1.737997,5.685945,1.737997,True,46,12
321,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__civilian_targeting_True,1.589535,4.901469,1.589535,True,46,12
159,logreg_l2_unweighted,full,1.0,0.0,none,cat__country_Sri Lanka,1.581760,4.863509,1.581760,True,46,12


In [7]:
coefs.sort_values('coefficient').head(20)

,variant,feature_profile,c_value,l1_ratio,class_weight,feature,coefficient,odds_ratio,abs_coefficient,selected_l1,numeric_feature_count,categorical_feature_count
314,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__country_Israel,-2.390826,0.091554,2.390826,True,46,12
472,logreg_l1_unweighted_c1,full,1.0,1.0,none,cat__country_Israel,-2.070573,0.126113,2.070573,True,46,12
317,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__country_infrequent_sklearn,-2.014272,0.133417,2.014272,True,46,12
318,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__country_Oman,-1.974433,0.138840,1.974433,True,46,12
0,logreg_l2_balanced,full,1.0,0.0,balanced,cat__country_Israel,-1.949732,0.142312,1.949732,True,46,12
158,logreg_l2_unweighted,full,1.0,0.0,none,cat__country_Israel,-1.716450,0.179703,1.716450,True,46,12
628,logreg_l1_balanced_c03,full,0.3,1.0,balanced,cat__country_Israel,-1.632818,0.195378,1.632818,True,46,12
474,logreg_l1_unweighted_c1,full,1.0,1.0,none,cat__country_infrequent_sklearn,-1.632059,0.195527,1.632059,True,46,12
320,logreg_l1_balanced_c1,full,1.0,1.0,balanced,cat__sub_event_type_Disrupted weapons use,-1.617208,0.198452,1.617208,True,46,12
475,logreg_l1_unweighted_c1,full,1.0,1.0,none,cat__country_Oman,-1.612392,0.199410,1.612392,True,46,12


## Seleccion L1

La regularizacion L1 deja coeficientes exactamente en cero. Para explicabilidad, miramos cuantas features sobreviven y revisamos el modelo `logreg_l1_balanced_c01` como candidato compacto.

In [8]:
selected_summary = (
    coefs[coefs['selected_l1']]
    .groupby('variant')
    .agg(selected_features=('feature', 'size'), max_abs_coefficient=('abs_coefficient', 'max'))
    .sort_values('selected_features')
)
selected_summary

,selected_features,max_abs_coefficient
variant,,
logreg_core_l1_unweighted_c01,20,0.761901
logreg_core_l1_balanced_c01,24,0.880482
logreg_l1_unweighted_c01,28,0.709313
logreg_l1_balanced_c01,33,0.748413
logreg_l1_unweighted_c03,54,1.137881
logreg_l1_balanced_c03,56,1.632818
logreg_l1_unweighted_c1,83,2.105714
logreg_l1_balanced_c1,90,2.390826
logreg_l2_balanced,155,1.949732


In [9]:
candidate = 'logreg_l1_balanced_c01'
coefs[(coefs['variant'].eq(candidate)) & (coefs['selected_l1'])][
    ['feature', 'coefficient', 'odds_ratio']
].sort_values('coefficient', ascending=False)

,feature,coefficient,odds_ratio
942,num__emb_pca_16,0.748413,2.113643
943,cat__target_type_civilian,0.694537,2.002781
945,cat__civilian_targeting_True,0.374660,1.454497
946,cat__sub_event_type_Air/drone strike,0.317122,1.373170
947,cat__target_type_unknown,0.294535,1.342502
948,num__is_airstrike,0.198577,1.219666
949,num__emb_pca_8,0.194889,1.215177
953,num__emb_pca_3,0.146320,1.157566
954,num__attacker_is_israel,0.138488,1.148536
957,num__emb_pca_10,0.123751,1.131734


## Auditoria de errores

Los falsos positivos y falsos negativos de mayo son la forma mas util de entender que esta aprendiendo el modelo.

In [10]:
temporal_predictions = predictions_df[predictions_df['split'].eq('temporal_train_until_apr_test_may')].copy()

false_positives = temporal_predictions[
    temporal_predictions['fatality_positive'].eq(0) & temporal_predictions['prediction_0_5'].eq(1)
].sort_values('fatality_probability', ascending=False)

false_negatives = temporal_predictions[
    temporal_predictions['fatality_positive'].eq(1) & temporal_predictions['prediction_0_5'].eq(0)
].sort_values('fatality_probability')

false_positives.head(10), false_negatives.head(10)

(                 event_id event_date       source country    weapon_type  \
 17      conflict_e4a51a41 2026-05-04   gdeltcloud    Iran  explosive_ied   
 155     conflict_c89e75ea 2026-05-20   gdeltcloud    Iraq          drone   
 1097    conflict_e4a51a41 2026-05-04   gdeltcloud    Iran  explosive_ied   
 1235    conflict_c89e75ea 2026-05-20   gdeltcloud    Iraq          drone   
 153   IRW-1779249784334-1 2026-05-20  iranwarlive    Iran        unknown   
 69    IRW-1778255444322-4 2026-05-08  iranwarlive    Iran     air_strike   
 105   IRW-1778608879489-0 2026-05-12  iranwarlive    Iran        unknown   
 695     conflict_c89e75ea 2026-05-20   gdeltcloud    Iraq          drone   
 102   IRW-1778616109178-1 2026-05-12  iranwarlive    Iran        unknown   
 1233  IRW-1779249784334-1 2026-05-20  iranwarlive    Iran        unknown   
 
          target_type  fatalities  target_msi  \
 17          civilian         0.0         0.0   
 155   infrastructure         0.0         0.0   
 109

## Persistencia

Para regenerar las salidas oficiales del proyecto, ejecutar desde la raiz:

```bash
python scripts/train_logreg_2026.py
```